Setup:

In [ ]:
import sys
import os
import plotly.graph_objects as go
import requests

from dotenv import load_dotenv
load_dotenv()

sys.path.insert(0, os.path.dirname(os.getcwd()))

Use decentralization.py to calculate Nakamoto, Gini, and HHI for each date in polygon_validators.csv.

In [58]:
import importlib
import src.collect.validators as val_module
import src.signals.decentralization as dec_module

importlib.reload(val_module)
importlib.reload(dec_module)

from src.signals.decentralization import compute_daily_metrics, save_metrics, load_validators

validators = load_validators()
metrics    = compute_daily_metrics(validators)
print(metrics.head(10))
print(metrics.describe())
save_metrics(metrics)

        date  nakamoto      gini         hhi
0 2024-01-01         6  0.741361  676.154798
1 2024-01-02         6  0.753476  637.357565
2 2024-01-03         6  0.759237  635.915287
3 2024-01-04         6  0.759264  630.123876
4 2024-01-05         6  0.768020  621.044630
5 2024-01-06         6  0.767674  600.828956
6 2024-01-07         6  0.767579  601.008546
7 2024-01-08         6  0.774140  599.377942
8 2024-01-09         6  0.775930  598.378539
9 2024-01-10         7  0.771770  578.151609
                      date    nakamoto        gini         hhi
count                  762  762.000000  762.000000  762.000000
mean   2025-01-15 12:00:00    6.741470    0.805918  570.400264
min    2024-01-01 00:00:00    5.000000    0.741361  493.377860
25%    2024-07-09 06:00:00    6.000000    0.783820  536.644927
50%    2025-01-15 12:00:00    7.000000    0.817899  562.177834
75%    2025-07-24 18:00:00    7.000000    0.820850  588.794209
max    2026-01-31 00:00:00    8.000000    0.835648  687.351092
s

Load the convergence windows on Polymarket and Kalshi for the trump_wins_2024 event.

In [60]:
import pandas as pd
from src.collect.polymarket import get_price_history as pm_get_ph
from src.collect.kalshi import get_price_history as kal_get_ph
from src.signals.decentralization import load_metrics

pm_df = pm_get_ph('trump_wins_2024')
kal_df = kal_get_ph('trump_wins_2024')
dec_df = load_metrics()

resolved_value = 1
window_hours = 48

resolution_time = pd.Timestamp('2024-11-06 06:00:00', tz='UTC')
window_start = resolution_time - pd.Timedelta(days=window_days)
window_end = resolution_time + pd.Timedelta(hours=window_hours)

print(f"Resolution: {resolution_time}")
print(f"Window start: {window_start}")
print(f"Window end: {window_end}")

pm_window = pm_df[(pm_df['timestamp'] >= window_start) & (pm_df['timestamp'] <= window_end)].copy()
kal_window = kal_df[(kal_df['timestamp'] >= window_start) & (kal_df['timestamp'] <= window_end)].copy()

print(f"\nPolymarket points in window: {len(pm_window)}")
print(f"Kalshi points in window: {len(kal_window)}")

print(f"\nPolymarket window: \n{pm_window.to_string()}")
print(f"\nKalshi window: \n{kal_window.to_string()}")

Resolution: 2024-11-06 06:00:00+00:00
Window start: 2024-10-30 06:00:00+00:00
Window end: 2024-11-08 06:00:00+00:00

Polymarket points in window: 15
Kalshi points in window: 9

Polymarket window: 
                    timestamp   price
599 2024-10-30 12:00:03+00:00  0.6685
600 2024-10-31 00:00:03+00:00  0.6475
601 2024-10-31 12:00:03+00:00  0.6485
602 2024-11-01 00:00:03+00:00  0.6325
603 2024-11-01 12:00:03+00:00  0.6285
604 2024-11-02 00:00:03+00:00  0.5825
605 2024-11-02 12:00:03+00:00  0.5795
606 2024-11-03 00:00:03+00:00  0.5585
607 2024-11-03 12:00:03+00:00  0.5585
608 2024-11-04 00:00:03+00:00  0.5635
609 2024-11-04 12:00:03+00:00  0.5705
610 2024-11-05 00:00:02+00:00  0.5855
611 2024-11-05 12:00:02+00:00  0.6225
612 2024-11-06 00:00:02+00:00  0.5800
613 2024-11-06 12:00:03+00:00  0.9985

Kalshi window: 
                   timestamp  price
26 2024-10-31 04:00:00+00:00   0.59
27 2024-11-01 04:00:00+00:00   0.58
28 2024-11-02 04:00:00+00:00   0.56
29 2024-11-03 04:00:00+00:00   0.4

Plot the points to visualize the convergence shape prior to computing the metric.

In [61]:
fig = go.Figure()

# polymarket price
fig.add_trace(go.Scatter(
  x=pm_window['timestamp'],
  y=pm_window['price'],
  mode='lines+markers',
  name='Polymarket',
  line=dict(color='royalblue', width=2)
))

# kalshi price
fig.add_trace(go.Scatter(
  x=kal_window['timestamp'],
  y=kal_window['price'],
  mode='lines+markers',
  name='Kalshi',
  line=dict(color='firebrick', width=2)
))

# resolved value line
fig.add_hline(
  y=resolved_value,
  line_dash='dash',
  line_color='green',
  annotation_text='Resolved value (1.0)',
  annotation_position='bottom right'
)

# resolution timestamp
fig.add_vline(
  x=resolution_time.timestamp() * 1000,
  line_dash='dash',
  line_color='grey',
  annotation_text='Resolution',
  annotation_position='top left'
)

fig.update_layout(
  title='Trump 2024 — Convergence Window',
  xaxis_title='Date',
  yaxis_title='Implied Probability',
  yaxis=dict(range=[0, 1], tickformat='.0%'),
  hovermode='x unified'
)

fig.show()

Compute convergence speed for the trump_wins_2024 event using trade-level data from the Polymarket Data API.

In [62]:
from src.collect.polymarket import get_price_history as pm_get_ph
from src.collect.kalshi import get_price_history as kal_get_ph
from src.signals.decentralization import load_metrics

# trump_wins_2024 metadata
condition_id = "0xdd22472e552920b8438158ea7238bfadfa4f736aa4cee91a6b86c39ead110917"
resolved_value = 1
resolution_time = pd.Timestamp('2024-11-06 06:00:00', tz='UTC')
window_hours = 48
window_days = 7

all_trades = []
offset = 0

while True:
  r = requests.get(
    "https://data-api.polymarket.com/trades",
    params = {
      "market": condition_id,
      "limit": 500,
      "offset": offset,
    }
  ).json()

  if not isinstance(r, list) or not r:
    break

  all_trades.extend(r)
  offset += 500

  if len(r) < 500:
    break

df_trades = pd.DataFrame([t for t in all_trades if isinstance(t, dict) and 'timestamp' in t])
df_trades = df_trades[df_trades['outcome'] == 'Yes'].copy()
df_trades['timestamp'] = pd.to_datetime(df_trades['timestamp'], unit='s', utc=True)
df_trades = df_trades[['timestamp', 'price']].sort_values('timestamp').reset_index(drop=True)

print(f"Total Yes trades fetched: {len(df_trades)}")
print(f"Date range: {df_trades['timestamp'].min()} to {df_trades['timestamp'].max()}")


Total Yes trades fetched: 3182
Date range: 2024-11-06 11:46:14+00:00 to 2024-11-06 15:20:35+00:00


Filter to post-resolution window and resample to hourly prices.

In [63]:
window_end  = resolution_time + pd.Timedelta(hours=window_hours)
post_trades = df_trades[
  (df_trades['timestamp'] >= resolution_time) &
  (df_trades['timestamp'] <= window_end)
].copy()

print(f"Post-resolution trades (48hr): {len(post_trades)}")

hourly = post_trades.set_index('timestamp')['price'].resample('1h').last().ffill().dropna()
hourly = hourly.reset_index()
hourly.columns = ['timestamp', 'price']

print(f"Hourly snapshots: {len(hourly)}")
print(hourly.to_string())

Post-resolution trades (48hr): 3182
Hourly snapshots: 5
                  timestamp  price
0 2024-11-06 11:00:00+00:00  0.998
1 2024-11-06 12:00:00+00:00  0.997
2 2024-11-06 13:00:00+00:00  0.997
3 2024-11-06 14:00:00+00:00  0.997
4 2024-11-06 15:00:00+00:00  0.998


Load Kalshi pre-resolution window for spread metric.

In [65]:
kal_df  = kal_get_ph('trump_wins_2024')

window_start = resolution_time - pd.Timedelta(days=window_days)
kal_pre = kal_df[
  (kal_df['timestamp'] >= window_start) &
  (kal_df['timestamp'] < resolution_time)
].copy()

print(f"Kalshi pre-resolution points: {len(kal_pre)}")
print(kal_pre.to_string())

Kalshi pre-resolution points: 7
                   timestamp  price
26 2024-10-31 04:00:00+00:00   0.59
27 2024-11-01 04:00:00+00:00   0.58
28 2024-11-02 04:00:00+00:00   0.56
29 2024-11-03 04:00:00+00:00   0.49
30 2024-11-04 04:00:00+00:00   0.53
31 2024-11-05 05:00:00+00:00   0.57
32 2024-11-06 05:00:00+00:00   0.95


Plot post-resolution convergence on Polymarket (hourly trades) and Kalshi.

In [66]:
kal_post = kal_df[
  (kal_df['timestamp'] >= resolution_time) &
  (kal_df['timestamp'] <= window_end)
].copy()

fig = go.Figure()

fig.add_trace(go.Scatter(
  x=hourly['timestamp'],
  y=hourly['price'],
  mode='lines+markers',
  name='Polymarket (hourly trades)',
  line=dict(color='royalblue', width=2)
))

fig.add_trace(go.Scatter(
  x=kal_post['timestamp'],
  y=kal_post['price'],
  mode='lines+markers',
  name='Kalshi',
  line=dict(color='firebrick', width=2)
))

fig.add_hline(
  y=resolved_value,
  line_dash='dash',
  line_color='green',
  annotation_text='Resolved value (1.0)',
  annotation_position='bottom right'
)

fig.add_vline(
  x=resolution_time.timestamp() * 1000,
  line_dash='dash',
  line_color='grey',
  annotation_text='Resolution',
  annotation_position='top left'
)

fig.update_layout(
  title='Trump 2024 — Post-Resolution Convergence (48hr)',
  xaxis_title='Date',
  yaxis_title='Implied Probability',
  yaxis=dict(range=[0, 1], tickformat='.0%'),
  hovermode='x unified'
)

fig.show()

Compute convergence speed, Kalshi spread, and pre-event Nakamoto coefficient.


In [68]:
# convergence speed
convergence_speed = float((hourly['price'] - resolved_value).abs().mean())

# kalshi spread
pm_df = pm_get_ph('trump_wins_2024')
pm_pre = pm_df[
  (pm_df['timestamp'] >= window_start) &
  (pm_df['timestamp'] < resolution_time)
].copy()

pm_daily = pm_pre.set_index('timestamp')['price'].resample('1D').last().dropna()
kal_daily = kal_pre.set_index('timestamp')['price'].resample('1D').last().dropna()
common = pm_daily.index.intersection(kal_daily.index)
kalshi_spread = float((pm_daily[common] - kal_daily[common]).abs().mean())

# nakamoto pre-event
dec_df = load_metrics()
naka_start = (resolution_time - pd.Timedelta(days=7)).date()
naka_end = resolution_time.date()
naka_mask = (dec_df['date'].dt.date >= naka_start) & (dec_df['date'].dt.date < naka_end)
nakamoto_pre = dec_df[naka_mask]['nakamoto'].mean()

# lower = faster convergence = more efficient
print(f"Convergence speed (Polymarket): {convergence_speed:.4f}")

# lower = closer to Kalshi benchmark = more efficient
print(f"Kalshi spread (pre-resolution): {kalshi_spread:.4f}")

# lower = more concentrated validator set
print(f"Nakamoto coefficient (pre-event): {nakamoto_pre:.2f}")

Convergence speed (Polymarket): 0.0026
Kalshi spread (pre-resolution): 0.0940
Nakamoto coefficient (pre-event): 7.00
